# The typed-config philosophy

Across the earlier notebooks you kept reaching for classes from `interactly.configs` —
`SayLLMNodeConfig`, `ConditionalEdgeConfig`, `WorkflowConfigFullyHydrated`, `OpenAILLMConfig`,
and friends. That was not incidental. The Interactly SDK is built around **typed objects**
rather than free-form dicts, and the companion `interactly-configs` package provides a
Pydantic model for **every** workflow config the server understands.

This notebook is a deep dive into *why that matters* and the leverage it gives you —
concepts that the task-focused notebooks used but never dwelt on:

1. **Validation at construction** — mistakes raise *before* any network call
2. **Discriminated unions** — `type` selects the right concrete class automatically
3. **Enums, not magic strings** — models, commands, node types are real enums
4. **Typed ⇄ dict duality** — every method takes either; responses hydrate back to types
5. **Composition** — configs nest into a whole fully-typed workflow graph
6. **One source of truth** — the typed class *is* the server's JSON Schema
7. **Typed API responses** — round-trip a config through the server and back

> Everything here works **without a server** except §7 — validation, unions, enums,
> serialisation, and schemas are all client-side. That is the point: the types catch
> errors on your machine, long before a request is sent.

> Requires the extra: `pip install "interactly[configs]"`. Without it the SDK still works —
> it just falls back to plain dicts (shown in §4).

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import interactly_configs

from interactly import AsyncWorkflowClient

client = AsyncWorkflowClient()
print("interactly-configs version:", interactly_configs.__version__)
print("Connected to", client._base_url)

## 1. Validation happens at construction — before any network call

A typed config validates its fields the moment you build it. A wrong type or a missing
required field raises a `pydantic.ValidationError` locally — you never send a malformed
request and wait for a 422 to find out.

In [ ]:
from pydantic import ValidationError

from interactly.configs import OpenAILLMConfig, OPENAIModel, SayLLMNodeConfig, PromptConfig

# A valid config builds fine.
good = OpenAILLMConfig(model=OPENAIModel.GPT_5_4, max_tokens=300, temperature=0.2)
print("valid:", good.model, good.max_tokens, good.temperature)

# An invalid one is rejected immediately — max_tokens must be an int, not a string.
try:
    OpenAILLMConfig(model=OPENAIModel.GPT_5_4, max_tokens="lots")
except ValidationError as e:
    print("\ncaught locally (no request sent):")
    print("  ", e.errors()[0]["loc"], "->", e.errors()[0]["msg"])

## 2. Discriminated unions — `type` picks the concrete class

`NodeConfig` and `EdgeConfig` are **discriminated unions** keyed on a `type` field. Hand a
raw dict to a `TypeAdapter` and Pydantic routes it to the right subclass; ask for the class
of a type name with `get_node_config_class`. Unknown types are rejected — the union is closed.

In [ ]:
from pydantic import TypeAdapter, ValidationError

from interactly.configs import NodeConfig, get_node_config_class

# A dict with type="say_llm" validates into SayLLMNodeConfig; type="say_static" into another.
n1 = TypeAdapter(NodeConfig).validate_python(
    {"type": "say_llm", "name": "A", "main_response_config": {"prompt": "hi"}}
)
n2 = TypeAdapter(NodeConfig).validate_python(
    {"type": "say_static", "name": "B", "static_messages_config": {"static_messages": ["hi"]}}
)
print("say_llm    ->", type(n1).__name__)
print("say_static ->", type(n2).__name__)

# Look up the concrete class for a type name directly.
print("get_node_config_class('say_llm') ->", get_node_config_class("say_llm").__name__)

# An unknown discriminator is rejected — you cannot smuggle in an unsupported node type.
try:
    TypeAdapter(NodeConfig).validate_python({"type": "telepathy_node"})
except ValidationError as e:
    print("\nunknown type rejected:", e.errors()[0]["msg"][:70], "...")

## 3. Enums, not magic strings

Model names, workflow commands, and node types are real `Enum`s. You get autocomplete and a
typo becomes an `AttributeError` in your editor instead of a silent runtime surprise. Each
member carries the wire `.value` the server expects.

In [ ]:
from interactly.configs import OPENAIModel, NodeType
from interactly import WorkflowCommand

print("OPENAIModel.GPT_5_4        ->", OPENAIModel.GPT_5_4.value)
print("WorkflowCommand.START      ->", WorkflowCommand.START.value)
print("NodeType.SAY_LLM           ->", NodeType.SAY_LLM.value)

print("\na few OpenAI models the SDK knows about:")
for m in list(OPENAIModel)[:6]:
    print("  ", m.name, "=", m.value)

## 4. Typed ⇄ dict duality

Every SDK method that takes a `*_config` accepts **either** a typed object or a plain dict —
internally it calls `serialise_config()` to normalise. And you can round-trip freely:
`model_dump()` to a dict, `model_validate()` back to a typed object. This is also the graceful
fallback: without the `[configs]` extra installed, the same methods still accept dicts.

In [ ]:
from interactly._utils._serialise import serialise_config
from interactly.configs import SayLLMNodeConfig, PromptConfig

typed = SayLLMNodeConfig(name="Greeter", main_response_config=PromptConfig(prompt="Hi!"))

# typed -> wire dict (exactly what the SDK sends)
as_dict = serialise_config(typed)
print("serialised keys:", sorted(k for k in as_dict if as_dict[k] is not None)[:6], "…")

# dict -> typed, and confirm the round-trip is faithful
roundtripped = SayLLMNodeConfig.model_validate(as_dict)
print("round-trip preserves prompt:",
      roundtripped.main_response_config.prompt == typed.main_response_config.prompt)
print("same type both ways:", type(roundtripped).__name__)

## 5. Configs compose into a whole typed graph

The typed models nest: a `WorkflowConfigFullyHydrated` holds `node_configs` and `edge_configs`,
each node nests a `PromptConfig` and an `OpenAILLMConfig`, an edge nests a `ConditionConfig`.
The entire workflow graph is one validated object tree — mistakes anywhere in it surface when
you build it, not at run time.

In [ ]:
from interactly.configs import (
    ConditionConfig, ConditionalEdgeConfig, DirectEdgeConfig, OpenAILLMConfig, OPENAIModel,
    PromptConfig, SayLLMNodeConfig, SayStaticMessageNodeConfig, StaticMessagesConfig,
    WorkflowConfig, WorkflowConfigFullyHydrated,
)

greet = SayLLMNodeConfig(name="Greet", is_start=True,
    main_response_config=PromptConfig(prompt="Greet the caller."),
    llms_config=OpenAILLMConfig(model=OPENAIModel.GPT_5_4_NANO, max_tokens=80))
bye = SayStaticMessageNodeConfig(name="Bye",
    static_messages_config=StaticMessagesConfig(static_messages=["Goodbye!"]))

graph = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(name="Typed Graph", category="System Examples"),
    node_configs=[greet, bye],
    edge_configs=[ConditionalEdgeConfig(
        source_node_logical_id=greet.logical_id,
        destination_node_logical_id=bye.logical_id,
        condition=ConditionConfig(condition_freeform="the caller says goodbye"))],
)

print("nodes:", [type(n).__name__ for n in graph.node_configs])
print("edges:", [type(e).__name__ for e in graph.edge_configs])
print("nested LLM on greet:", type(greet.llms_config).__name__, greet.llms_config.model.value)
print("whole graph is one model:", type(graph).__name__)

## 6. One source of truth — the typed class *is* the server schema

The typed classes are not a hand-maintained mirror that can drift. Each model emits its own
JSON Schema via `model_json_schema()`, and that is the very schema the server serves from
`client.nodes.schema(<type>)`. Same fields, one definition.

In [ ]:
from interactly.configs import SayLLMNodeConfig

# Schema straight from the typed class (client-side)...
local_props = set(SayLLMNodeConfig.model_json_schema().get("properties", {}))

# ...and the schema the server advertises for the same node type.
server_props = set((await client.nodes.schema("say_llm")).get("config_schema", {}).get("properties", {}))

print("fields on the typed class :", len(local_props))
print("fields in the server schema:", len(server_props))
print("shared fields (sample)    :", sorted(local_props & server_props)[:8])

## 7. Typed API responses

The duality runs both directions: send a typed config, and the SDK hydrates the **response**
back into the same typed class (when `[configs]` is installed). `node.node_config` comes back
as a `SayLLMNodeConfig`, not a dict — so you keep autocomplete and validation on data the
server returns, too.

In [ ]:
from interactly.configs import BaseNodeConfig, SayLLMNodeConfig, PromptConfig
from interactly.types.nodes.node import Node
from interactly.types.workflows.workflow import Workflow

workflow: Workflow = await client.workflows.create(name="Typed Config Deep Dive")
WF_ID = workflow.id

created: Node = await client.nodes.create(node_config=SayLLMNodeConfig(
    workflow_id=WF_ID, name="Greet User", is_start=True,
    main_response_config=PromptConfig(prompt="Greet the user warmly."),
))

fetched: Node = await client.nodes.get(created.id)
cfg = fetched.node_config
print("node_config type:", type(cfg).__name__)
print("is a BaseNodeConfig:", isinstance(cfg, BaseNodeConfig))
if isinstance(cfg, BaseNodeConfig):
    print("typed access -> name:", cfg.name, "| prompt:", cfg.main_response_config.prompt)

## Cleanup

In [ ]:
await client.workflows.delete(WF_ID)
await client.close()
print("cleaned up")

## See also

- [`16_nodes_and_edges.ipynb`](16_nodes_and_edges.ipynb) — the typed configs used to build a graph server-side
- [`02_interactive_workflow.ipynb`](02_interactive_workflow.ipynb) — typed configs assembled with `create_from_config`